# Test stockgen-upscaler
Test the upscaling engine on Colab before RunPod deploy.

Runtime → Change runtime → T4 GPU required.

In [ ]:
import os, subprocess, urllib.request, cv2, torch
from google.colab.patches import cv2_imshow

In [ ]:
# Install realesrgan
!pip install -q opencv-python-headless basicsr realesrgan

In [ ]:
# Patch basicsr for torchvision
import importlib.util, os
spec = importlib.util.find_spec('basicsr')
path = os.path.join(os.path.dirname(spec.origin), 'data', 'degradations.py')
content = open(path).read()
content = content.replace(
    'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
    'from torchvision.transforms.functional import rgb_to_grayscale'
)
open(path, 'w').write(content)
print('Patched')

In [ ]:
# Download weights
os.makedirs('/weights', exist_ok=True)
urls = [
    'https://huggingface.co/nateraw/real-esrgan/resolve/main/RealESRGAN_x2plus.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
]
for url in urls:
    name = url.split('/')[-1]
    path = f'/weights/{name}'
    if not os.path.exists(path):
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, path)
    else:
        print(f'{name} exists')

In [ ]:
# Test x4 upscale
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4, model_path='/weights/RealESRGAN_x4plus.pth',
    model=model, tile=400, tile_pad=10, pre_pad=0, half=True, gpu_id=0,
)
print(f'x4 model loaded on {device}')

In [ ]:
# Download a test image and upscale
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/xinntao/Real-ESRGAN/master/assets/butterfly.png',
    '/content/input.png'
)
img = cv2.imread('/content/input.png', cv2.IMREAD_COLOR)
print(f'Input: {img.shape[1]}x{img.shape[0]}')
output, _ = upsampler.enhance(img, outscale=4)
print(f'Output: {output.shape[1]}x{output.shape[0]}')
cv2.imwrite('/content/output.png', output)
print('✓ Upscale OK')

In [ ]:
# Show results
print('Input:')
cv2_imshow(cv2.resize(img, (300, 225)))
print('Output:')
cv2_imshow(cv2.resize(output, (600, 450)))